In [49]:
import sys
!{sys.executable} -m pip install pytorch-crf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: no such file or directory: /Users/mohamedelsayed/Documents/MDS/Deep


In [ ]:
import sys
import os


!{sys.executable} -m pip install pytorch-crf


import site
site.addsitedir(os.path.join(os.path.dirname(sys.executable), "..", "lib", "python3.10", "site-packages"))


try:
    from torchcrf import CRF
    print("✅ ")
except ImportError:
    print("❌ ")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: no such file or directory: /Users/mohamedelsayed/Documents/MDS/Deep
✅ 


In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer
from torchcrf import CRF
import numpy as np
import random
import time
import re
from seqeval.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [ ]:

# ============================================================
# PHASE 1: DATA ENGINEERING & MAPPING
# ============================================================

def load_tsv_data(file_path):
    sentences, labels = [], []
    cur_toks, cur_labs = [], []

    def flush():
        nonlocal cur_toks, cur_labs
        if cur_toks:
            sentences.append(cur_toks)
            labels.append(cur_labs)
            cur_toks, cur_labs = [], []

    with open(file_path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.rstrip('\n')

            # sentence boundary
            if not line.strip():
                flush()
                continue

            # comment/metadata/doc boundary
            # valid token lines in this dataset start like: "1\tToken\tLabel..."
            if line.startswith('#') or not re.match(r'^\d+\t', line):
                flush()
                continue

            parts = line.split('\t')
            if len(parts) >= 3:
                token = parts[1]
                label = parts[2]
                cur_toks.append(token)
                cur_labs.append(label)

    flush()
    return sentences, labels

def clean_labels_strategic(labels):
    """
    Correct mapping for this task:
    - Keep only PER / ORG / LOC
    - Remove part, deriv, OTH completely
    """
    cleaned = []
    for sent_labels in labels:
        cleaned_sent = []
        for label in sent_labels:
            if label == 'O':
                cleaned_sent.append('O')

            elif 'part' in label:
                cleaned_sent.append('O')  

            elif 'deriv' in label or 'OTH' in label:
                cleaned_sent.append('O')

            else:
                cleaned_sent.append(label)  # B-PER, I-PER, B-ORG, ...
        cleaned.append(cleaned_sent)
    return cleaned


# Load data
DATA_DIR = "/Users/mohamedelsayed/Documents/MDS/Deep learning/task-4/public_data-4"
train_sentences, train_labels = load_tsv_data(f'{DATA_DIR}/train.tsv')
val_sentences, val_labels = load_tsv_data(f'{DATA_DIR}/val.tsv')

train_labels_clean = clean_labels_strategic(train_labels)
val_labels_clean = clean_labels_strategic(val_labels)

label2id = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6}
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)

print(f"Labels: {list(label2id.keys())}")
print(f"Train: {len(train_sentences)} sentences")
print(f"Val: {len(val_sentences)} sentences")


Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
Train: 24000 sentences
Val: 2200 sentences


In [53]:
print("Train sentences:", len(train_sentences))
print("Val sentences:", len(val_sentences))

print("\nFirst train sentence:")
print(list(zip(train_sentences[0][:10], train_labels[0][:10])))

print("\nFirst val sentence:")
print(list(zip(val_sentences[0][:10], val_labels[0][:10])))


Train sentences: 24000
Val sentences: 2200

First train sentence:
[('Schartau', 'B-PER'), ('sagte', 'O'), ('dem', 'O'), ('"', 'O'), ('Tagesspiegel', 'B-ORG'), ('"', 'O'), ('vom', 'O'), ('Freitag', 'O'), (',', 'O'), ('Fischer', 'B-PER')]

First val sentence:
[('Gleich', 'O'), ('darauf', 'O'), ('entwirft', 'O'), ('er', 'O'), ('seine', 'O'), ('Selbstdarstellung', 'O'), ('"', 'O'), ('Ecce', 'B-OTH'), ('homo', 'I-OTH'), ('"', 'O')]


In [54]:
def load_tsv_data(file_path):
    sentences, labels = [], []
    cur_toks, cur_labs = [], []

    def flush():
        nonlocal cur_toks, cur_labs
        if cur_toks:
            sentences.append(cur_toks)
            labels.append(cur_labs)
            cur_toks, cur_labs = [], []

    with open(file_path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.rstrip('\n')

            # sentence boundary
            if not line.strip():
                flush()
                continue

            # comment/metadata/doc boundary
            # valid token lines in this dataset start like: "1\tToken\tLabel..."
            if line.startswith('#') or not re.match(r'^\d+\t', line):
                flush()
                continue

            parts = line.split('\t')
            if len(parts) >= 3:
                token = parts[1]
                label = parts[2]
                cur_toks.append(token)
                cur_labs.append(label)

    flush()
    return sentences, labels

In [55]:

# ============================================================
# PHASE 2: FEATURE EXTRACTION (FROZEN ENCODER)
# ============================================================

class FrozenFeatureExtractor(nn.Module):
    """Extract features from frozen GELECTRA with layer pooling"""
    
    def __init__(self, model_name='deepset/gelectra-large'):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Freeze all parameters
        for param in self.model.parameters():
            param.requires_grad = False
        self.model.eval()
        
        self.n_layers = 4
        self.hidden_size = self.model.config.hidden_size
        self.feature_dim = self.hidden_size * self.n_layers
    
    @torch.no_grad()
    def forward(self, input_ids, attention_mask):
        """Extract features with layer pooling"""
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        # Concatenate last 4 layers
        hidden_states = outputs.hidden_states[-self.n_layers:]
        features = torch.cat(hidden_states, dim=-1)
        
        return features


In [56]:

# ============================================================
# PHASE 3: TRAINABLE HEAD WITH WEIGHTED HYBRID LOSS
# ============================================================

class TrainableNERHeadWeighted(nn.Module):
    """Custom head with word-level compression + WEIGHTED HYBRID LOSS"""
    
    def __init__(self, input_dim=4096, hidden_dim=256, num_labels=7, dropout=0.5):
        super().__init__()
        
        self.projection = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.lstm = nn.LSTM(
            512, hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )
        
        self.dropouts = nn.ModuleList([nn.Dropout(dropout) for _ in range(5)])
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        
        # ✨ WEIGHTED CE LOSS
        # O=1.0, B-PER=2.5, I-PER=1.0, B-ORG=3.0, I-ORG=1.0, B-LOC=2.5, I-LOC=1.0
        self.ce_weights = torch.tensor([1.3, 2.5, 1.0, 3.0, 1.0, 2.5, 1.0])
        
        # Hybrid loss ratio
        self.crf_weight = 1
        self.ce_weight = 0

    def compress_to_word_level(self, features, word_indices, max_words: int):
        """Compress subword features -> word-level features"""
        B, S, D = features.shape
        device = features.device
        max_words = max(int(max_words), 1)

        word_features = torch.zeros(B, max_words, D, device=device)
        auto_mask = torch.zeros(B, max_words, dtype=torch.bool, device=device)

        for i in range(B):
            seen = set()
            for j in range(S):
                w = int(word_indices[i, j].item())
                if w >= 0 and w < max_words and w not in seen:
                    word_features[i, w] = features[i, j]
                    auto_mask[i, w] = True
                    seen.add(w)

        return word_features, auto_mask
    
    def forward(self, features, word_indices, word_labels=None, word_mask=None, return_emissions=False):
        """Forward with WEIGHTED HYBRID LOSS"""
        # Determine word length
        if word_labels is not None:
            W = word_labels.size(1)
        elif word_mask is not None:
            W = word_mask.size(1)
        else:
            valid = word_indices[word_indices >= 0]
            W = int(valid.max().item()) + 1 if valid.numel() else 1

        # Compress to word-level
        word_features, auto_mask = self.compress_to_word_level(features, word_indices, max_words=W)

        if word_mask is None:
            word_mask = auto_mask
        else:
            word_mask = word_mask.to(dtype=torch.bool, device=features.device)

        word_features = word_features * word_mask.unsqueeze(-1)

        # Projection + LSTM
        x = self.projection(word_features)
        x, _ = self.lstm(x)

        # Logits with multi-sample dropout
        if self.training:
            logits = torch.stack([self.classifier(d(x)) for d in self.dropouts], dim=0).mean(dim=0)
        else:
            logits = self.classifier(x)

        # Training: WEIGHTED HYBRID LOSS
        if word_labels is not None:
            # 1. CRF loss
            crf_loss = -self.crf(logits, word_labels, mask=word_mask, reduction='mean')
            
            # 2. Weighted CE loss
            ce_weights = self.ce_weights.to(logits.device)
            ce_loss_fn = nn.CrossEntropyLoss(weight=ce_weights, reduction='none')
            
            # Flatten for CE
            logits_flat = logits.view(-1, logits.size(-1))
            labels_flat = word_labels.view(-1)
            mask_flat = word_mask.view(-1)
            
            ce_loss = ce_loss_fn(logits_flat, labels_flat)
            ce_loss = (ce_loss * mask_flat).sum() / mask_flat.sum()
            
            # 3. HYBRID LOSS: 85% CRF + 15% Weighted CE
            hybrid_loss = self.crf_weight * crf_loss + self.ce_weight * ce_loss
            
            if return_emissions:
                return hybrid_loss, logits
            return hybrid_loss, logits
        else:
            # Inference
            if return_emissions:
                predictions = self.crf.decode(logits, mask=word_mask)
                return predictions, logits
            return self.crf.decode(logits, mask=word_mask)


In [57]:

# ============================================================
# AUGMENTATION
# ============================================================

class EntityAugmenter:
    """Entity swapping with same-length constraint + token dropout"""
    
    def __init__(self, sentences, labels, swap_prob=0.15, dropout_prob=0.1):
        self.swap_prob = swap_prob
        self.dropout_prob = dropout_prob
        
        # Collect entities by type and length
        self.entity_pool = {'PER': {}, 'ORG': {}, 'LOC': {}}
        
        for sent_tokens, sent_labels in zip(sentences, labels):
            entity = []
            entity_type = None
            
            for token, label in zip(sent_tokens, sent_labels):
                if label.startswith('B-'):
                    if entity and entity_type:
                        length = len(entity)
                        if length not in self.entity_pool[entity_type]:
                            self.entity_pool[entity_type][length] = []
                        self.entity_pool[entity_type][length].append(entity)
                    entity = [token]
                    entity_type = label[2:]
                elif label.startswith('I-') and entity:
                    entity.append(token)
                else:
                    if entity and entity_type in self.entity_pool:
                        length = len(entity)
                        if length not in self.entity_pool[entity_type]:
                            self.entity_pool[entity_type][length] = []
                        self.entity_pool[entity_type][length].append(entity)
                    entity = []
                    entity_type = None
            
            if entity and entity_type in self.entity_pool:
                length = len(entity)
                if length not in self.entity_pool[entity_type]:
                    self.entity_pool[entity_type][length] = []
                self.entity_pool[entity_type][length].append(entity)
    
    def augment(self, tokens, labels):
        """Apply entity swapping + token dropout"""
        new_tokens = tokens.copy()
        
        if random.random() < self.swap_prob:
            i = 0
            while i < len(labels):
                if labels[i].startswith('B-'):
                    entity_type = labels[i][2:]
                    
                    j = i + 1
                    while j < len(labels) and labels[j] == f'I-{entity_type}':
                        j += 1
                    
                    entity_length = j - i
                    
                    if (entity_type in self.entity_pool and 
                        entity_length in self.entity_pool[entity_type] and
                        len(self.entity_pool[entity_type][entity_length]) > 1):
                        
                        replacement = random.choice(self.entity_pool[entity_type][entity_length])
                        
                        for k in range(entity_length):
                            new_tokens[i + k] = replacement[k]
                    
                    i = j
                else:
                    i += 1
        
        if random.random() < self.dropout_prob:
            for i in range(len(new_tokens)):
                if labels[i] == 'O' and random.random() < 0.1:
                    new_tokens[i] = '[UNK]'
        
        return new_tokens, labels

In [58]:

# ============================================================
# DATASET WITH WORD-LEVEL ALIGNMENT
# ============================================================

class NERDataset(Dataset):
    """Dataset with word-level labels and indices"""
    
    def __init__(self, sentences, labels, tokenizer, label2id, augmenter=None, is_train=False):
        self.sentences = sentences
        self.labels = labels
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.augmenter = augmenter
        self.is_train = is_train
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        tokens = self.sentences[idx]
        labels = self.labels[idx]
        
        if self.augmenter and self.is_train:
            tokens, labels = self.augmenter.augment(tokens, labels)
        
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_tensors='pt',
            truncation=True,
            max_length=128
        )
        
        word_ids = encoding.word_ids(0)
        word_indices = []
        word_labels = []
        seen_words = set()
        
        for word_id in word_ids:
            if word_id is None:
                word_indices.append(-1)
            else:
                word_indices.append(int(word_id))
                if word_id not in seen_words:
                    lab = labels[word_id] if labels[word_id] in self.label2id else "O"
                    word_labels.append(self.label2id[lab])
                    seen_words.add(word_id)
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'word_indices': torch.tensor(word_indices, dtype=torch.long),
            'word_labels': torch.tensor(word_labels, dtype=torch.long)
        }

def collate_fn(batch, pad_token_id):
    """Custom collate with proper padding"""
    max_subword_len = max(item['input_ids'].size(0) for item in batch)
    max_word_len = max(item['word_labels'].size(0) for item in batch)
    
    word_lens = torch.tensor([item['word_labels'].size(0) for item in batch], dtype=torch.long)
    
    input_ids = torch.stack([
        F.pad(item['input_ids'], (0, max_subword_len - item['input_ids'].size(0)), value=pad_token_id)
        for item in batch
    ])
    
    attention_mask = torch.stack([
        F.pad(item['attention_mask'], (0, max_subword_len - item['attention_mask'].size(0)), value=0)
        for item in batch
    ])
    
    word_indices = torch.stack([
        F.pad(item['word_indices'], (0, max_subword_len - item['word_indices'].size(0)), value=-1)
        for item in batch
    ])
    
    word_labels = torch.stack([
        F.pad(item['word_labels'], (0, max_word_len - item['word_labels'].size(0)), value=0)
        for item in batch
    ])
    
    word_mask = (torch.arange(max_word_len).unsqueeze(0) < word_lens.unsqueeze(1)).to(torch.bool)
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'word_indices': word_indices,
        'word_labels': word_labels,
        'word_mask': word_mask
    }


In [59]:
# ============================================================
# TRAINING SETUP
# ============================================================

feature_extractor = FrozenFeatureExtractor()
feature_extractor.to(device)



# =========================
# DEBUG: TOKENIZATION ALIGNMENT CHECK
# =========================
idx = 0
tokens = val_sentences[idx]
labs   = val_labels_clean[idx]

tok = feature_extractor.tokenizer

enc = tok(
    tokens,
    is_split_into_words=True,
    return_tensors='pt',
    truncation=True,
    max_length=128
)

word_ids = enc.word_ids(0)
sub_tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])

print("SUBWORD TOKENS WITH word_id:")
for t, wid in list(zip(sub_tokens, word_ids))[:60]:
    print(f"{t:15s}  wid={wid}")

print("\nORIGINAL TOKENS (first 30):")
for i, (w, l) in enumerate(list(zip(tokens, labs))[:30]):
    print(i, w, l)

# simulate your current dataset logic
word_labels = []
seen = set()
for wid in word_ids:
    if wid is None:
        continue
    if wid not in seen:
        lab = labs[wid] if labs[wid] in label2id else "O"
        word_labels.append(label2id[lab])
        seen.add(wid)

print("\nProduced word_labels length:", len(word_labels))
print("First 30 produced labels:", [id2label[x] for x in word_labels[:30]])
print("First 30 gold labels:    ", labs[:30])

augmenter = EntityAugmenter(train_sentences, train_labels_clean, swap_prob=0.15, dropout_prob=0.1)

train_dataset = NERDataset(train_sentences, train_labels_clean, feature_extractor.tokenizer, 
                           label2id, augmenter, is_train=True)
val_dataset = NERDataset(val_sentences, val_labels_clean, feature_extractor.tokenizer, 
                         label2id, is_train=False)

def get_sentence_weights(labels):
    weights = []
    for sent_labels in labels:
        has_entity = any(label != 'O' for label in sent_labels)
        weights.append(2.0 if has_entity else 1.0)
    return weights

#train_weights = get_sentence_weights(train_labels_clean)
#sampler = WeightedRandomSampler(train_weights, len(train_weights))

pad_token_id = feature_extractor.tokenizer.pad_token_id
train_loader = DataLoader(
    train_dataset, batch_size=8, shuffle=True, 
    collate_fn=lambda b: collate_fn(b, pad_token_id), num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_token_id), num_workers=0
)

# ✨ NEW MODEL WITH WEIGHTED LOSS
model = TrainableNERHeadWeighted(input_dim=4096, num_labels=num_labels)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4, steps_per_epoch=len(train_loader), epochs=10
)

print(f"\n✨ NEW: Weighted Hybrid Loss Training")
print(f"  CE Weights: O=1.0, B-PER=2.5, I-PER=1.0, B-ORG=3.0, I-ORG=1.0, B-LOC=2.5, I-LOC=1.0")
print(f"  Loss: 0.85×CRF + 0.15×Weighted_CE")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

SUBWORD TOKENS WITH word_id:
[CLS]            wid=None
Gleich           wid=0
darauf           wid=1
entw             wid=2
##ir             wid=2
##ft             wid=2
er               wid=3
seine            wid=4
Selbst           wid=5
##dar            wid=5
##stellung       wid=5
"                wid=6
Ec               wid=7
##ce             wid=7
hom              wid=8
##o              wid=8
"                wid=9
in               wid=10
enger            wid=11
Auseinander      wid=12
##setzung        wid=12
mit              wid=13
diesem           wid=14
Bild             wid=15
Jesu             wid=16
.                wid=17
[SEP]            wid=None

ORIGINAL TOKENS (first 30):
0 Gleich O
1 darauf O
2 entwirft O
3 er O
4 seine O
5 Selbstdarstellung O
6 " O
7 Ecce O
8 homo O
9 " O
10 in O
11 enger O
12 Auseinandersetzung O
13 mit O
14 diesem O
15 Bild O
16 Jesu B-PER
17 . O

Produced word_labels length: 18
First 30 produced labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O

In [60]:

# ============================================================
# ENHANCED TRAINING & EVALUATION
# ============================================================

def train_epoch(model, feature_extractor, loader, optimizer, scheduler):
    """Training epoch with detailed metrics"""
    model.train()
    feature_extractor.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        word_indices = batch['word_indices'].to(device)
        word_labels = batch['word_labels'].to(device)
        word_mask = batch['word_mask'].to(device)
        
                # =========================
        # DEBUG: check missing word positions (FIRST BATCH ONLY)
        # =========================
        if total_loss == 0:
            i = 0  # first sample in batch
            wi = word_indices[i].detach().cpu()
            wm = word_mask[i].detach().cpu()
            true_len = int(wm.sum().item())

            missing = 0
            for k in range(true_len):
                js = (wi == k).nonzero(as_tuple=True)[0]
                if js.numel() == 0:
                    missing += 1

            print("[DEBUG] true_len:", true_len,
                "| missing_word_positions:", missing)


        with torch.no_grad():
            features = feature_extractor(input_ids, attention_mask)
        
        loss, logits = model(features=features, word_indices=word_indices, 
                            word_labels=word_labels, word_mask=word_mask)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        # Get predictions for train metrics
        with torch.no_grad():
            predictions = model(features=features, word_indices=word_indices, 
                              word_labels=None, word_mask=word_mask)
            
            for pred_seq, label_seq, mask_seq in zip(predictions, word_labels, word_mask):
                actual_len = mask_seq.sum().item()
                pred_labels = [id2label[p] for p in pred_seq[:actual_len]]
                true_labels = [id2label[l.item()] for l in label_seq[:actual_len]]
                
                if pred_labels and true_labels:
                    all_preds.append(pred_labels)
                    all_labels.append(true_labels)
    
    # Calculate train F1
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    
    f1_scores = []
    entity_f1s = {}
    for entity in ['PER', 'ORG', 'LOC']:
        if entity in report:
            f1 = report[entity]['f1-score']
            f1_scores.append(f1)
            entity_f1s[entity] = f1
        else:
            f1_scores.append(0.0)
            entity_f1s[entity] = 0.0
    
    macro_f1 = np.mean(f1_scores)
    
    return total_loss / len(loader), macro_f1, entity_f1s

def evaluate_detailed(model, feature_extractor, loader, id2label):
    """Detailed evaluation with per-entity metrics and FP count"""
    model.eval()
    feature_extractor.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            word_indices = batch['word_indices'].to(device)
            word_labels = batch['word_labels'].to(device)
            word_mask = batch['word_mask'].to(device)
            
            features = feature_extractor(input_ids, attention_mask)
            predictions = model(features=features, word_indices=word_indices, 
                              word_labels=None, word_mask=word_mask)
            
            for pred_seq, label_seq, mask_seq in zip(predictions, word_labels, word_mask):
                actual_len = mask_seq.sum().item()
                
                pred_labels = [id2label[p] for p in pred_seq[:actual_len]]
                true_labels = [id2label[l.item()] for l in label_seq[:actual_len]]
                
                if pred_labels and true_labels:
                    all_preds.append(pred_labels)
                    all_labels.append(true_labels)
    
    # Calculate metrics
    report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
    
    f1_scores = []
    entity_metrics = {}
    for entity in ['PER', 'ORG', 'LOC']:
        if entity in report:
            entity_metrics[entity] = {
                'precision': report[entity]['precision'],
                'recall': report[entity]['recall'],
                'f1': report[entity]['f1-score'],
                'support': report[entity]['support']
            }
            f1_scores.append(report[entity]['f1-score'])
        else:
            entity_metrics[entity] = {
                'precision': 0.0,
                'recall': 0.0,
                'f1': 0.0,
                'support': 0
            }
            f1_scores.append(0.0)
    
    macro_f1 = np.mean(f1_scores)
    
    # Count FP (entity-level)
    fp_count = 0
    for pred_sent, true_sent in zip(all_preds, all_labels):
        pred_entities = extract_entity_spans(pred_sent)
        true_entities = extract_entity_spans(true_sent)
        fp_count += len(pred_entities - true_entities)
    
    return macro_f1, entity_metrics, fp_count, all_preds, all_labels

def extract_entity_spans(labels):
    """Extract entity spans as (type, start, end) tuples"""
    spans = set()
    i = 0
    while i < len(labels):
        if labels[i].startswith('B-'):
            entity_type = labels[i][2:]
            start = i
            i += 1
            while i < len(labels) and labels[i] == f'I-{entity_type}':
                i += 1
            spans.add((entity_type, start, i))
        else:
            i += 1
    return spans

In [61]:

# TRAINING LOOP WITH ENHANCED LOGGING
# ============================================================

print("\n" + "="*80)
print("🚀 TRAINING WITH WEIGHTED HYBRID LOSS")
print("="*80)

best_val_f1 = 0
patience = 3
patience_counter = 0

training_history = []

for epoch in range(10):
    epoch_start = time.time()
    
    # Train
    train_loss, train_f1, train_entity_f1s = train_epoch(
        model, feature_extractor, train_loader, optimizer, scheduler
    )
    
    # Validate
    val_f1, val_entity_metrics, val_fp_count, _, _ = evaluate_detailed(
        model, feature_extractor, val_loader, id2label
    )
    
    epoch_time = time.time() - epoch_start
    
    # Log
    print(f"\n{'='*80}")
    print(f"📊 EPOCH {epoch + 1}/10 (Time: {epoch_time:.1f}s)")
    print(f"{'='*80}")
    
    print(f"\n🏋️  TRAIN:")
    print(f"  Loss: {train_loss:.4f}")
    print(f"  Macro-F1: {train_f1:.4f}")
    print(f"  Per-entity F1:")
    for entity in ['PER', 'ORG', 'LOC']:
        print(f"    {entity}: {train_entity_f1s[entity]:.4f}")
    
    print(f"\n✅ VALIDATION:")
    print(f"  Macro-F1: {val_f1:.4f}")
    print(f"  FP Count (entity-level): {val_fp_count}")
    print(f"\n  Detailed Metrics:")
    print(f"  {'Entity':<8} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print(f"  {'-'*60}")
    for entity in ['PER', 'ORG', 'LOC']:
        m = val_entity_metrics[entity]
        print(f"  {entity:<8} {m['precision']:<12.4f} {m['recall']:<12.4f} "
              f"{m['f1']:<12.4f} {int(m['support']):<10d}")
    
    # Save best
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), 'best_head_weighted.pt')
        torch.save({
            'head_state': model.state_dict(),
            'label2id': label2id,
            'id2label': id2label,
            'val_f1': val_f1,
            'val_metrics': val_entity_metrics
        }, 'complete_model_weighted.pt')
        print(f"\n  🏆 NEW BEST! Val F1: {best_val_f1:.4f} → Saved!")
    else:
        patience_counter += 1
        print(f"\n  ⏸️  No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\n⚠️  Early stopping at epoch {epoch + 1}")
            break
    
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'val_fp': val_fp_count
    })
    
    if device.type == "mps":
        torch.mps.empty_cache()

print(f"\n{'='*80}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*80}")
print(f"\n🏆 Best Val F1: {best_val_f1:.4f}")
print(f"\n📈 Training Progress:")
for h in training_history:
    print(f"  Epoch {h['epoch']}: Train F1={h['train_f1']:.4f}, "
          f"Val F1={h['val_f1']:.4f}, Val FP={h['val_fp']}")

# Save training history
torch.save(training_history, 'training_history_weighted.pt')


🚀 TRAINING WITH WEIGHTED HYBRID LOSS
[DEBUG] true_len: 11 | missing_word_positions: 0

📊 EPOCH 1/10 (Time: 1599.2s)

🏋️  TRAIN:
  Loss: 2.9824
  Macro-F1: 0.6112
  Per-entity F1:
    PER: 0.7395
    ORG: 0.4594
    LOC: 0.6347

✅ VALIDATION:
  Macro-F1: 0.8375
  FP Count (entity-level): 280

  Detailed Metrics:
  Entity   Precision    Recall       F1-Score     Support   
  ------------------------------------------------------------
  PER      0.9289       0.9367       0.9328       711       
  ORG      0.7165       0.7440       0.7300       496       
  LOC      0.8300       0.8702       0.8496       763       

  🏆 NEW BEST! Val F1: 0.8375 → Saved!
[DEBUG] true_len: 10 | missing_word_positions: 0

📊 EPOCH 2/10 (Time: 1652.3s)

🏋️  TRAIN:
  Loss: 0.8543
  Macro-F1: 0.8524
  Per-entity F1:
    PER: 0.9335
    ORG: 0.7462
    LOC: 0.8777

✅ VALIDATION:
  Macro-F1: 0.8663
  FP Count (entity-level): 221

  Detailed Metrics:
  Entity   Precision    Recall       F1-Score     Support   
  -

KeyboardInterrupt: 

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "deepset/gelectra-large"
ENC_DIR = "submission/weights/encoder"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

encoder = AutoModel.from_pretrained(MODEL_NAME)
encoder.eval()
encoder.half()  

encoder.save_pretrained(ENC_DIR, safe_serialization=True)
tokenizer.save_pretrained(ENC_DIR)

print("✅ Saved FP16 encoder to:", ENC_DIR)


✅ Saved FP16 encoder to: submission/weights/encoder


In [ ]:

size_mb = os.path.getsize("submission.zip") / (1024**2)
print(f"submission.zip size: {size_mb:.2f} MB")


submission.zip size: 603.20 MB


In [95]:
# ===== Reload best head weights (no training) =====
model.load_state_dict(torch.load("best_head_weighted.pt", map_location=device))
model.eval()
feature_extractor.eval()

# ===== Run evaluation again to get all_preds / all_labels =====
val_f1, val_entity_metrics, val_fp_count, all_preds, all_labels = evaluate_detailed(
    model, feature_extractor, val_loader, id2label
)

print("Val Macro-F1:", val_f1)
print("Val FP (entity-level):", val_fp_count)
print(val_entity_metrics)


Val Macro-F1: 0.9006226827800217
Val FP (entity-level): 168
{'PER': {'precision': 0.9424657534246575, 'recall': 0.9676511954992968, 'f1': 0.9548924358084664, 'support': 711}, 'ORG': {'precision': 0.8322981366459627, 'recall': 0.8104838709677419, 'f1': 0.8212461695607762, 'support': 496}, 'LOC': {'precision': 0.9369127516778524, 'recall': 0.9148099606815203, 'f1': 0.9257294429708224, 'support': 763}}


In [68]:
from collections import Counter

def analyze_false_positives(all_preds, all_labels, all_tokens):
    fp_words = Counter()
    fp_by_type = Counter()

    for preds, golds, toks in zip(all_preds, all_labels, all_tokens):
        for p, g, w in zip(preds, golds, toks):
            if p != 'O' and g == 'O':
                fp_words[w] += 1
                fp_by_type[p[2:]] += 1

    print("\n🔴 Top False Positive Tokens:")
    for w, c in fp_words.most_common(30):
        print(f"{w:<25} {c}")

    print("\n📊 FP by Entity Type:")
    for k, v in fp_by_type.most_common():
        print(f"{k}: {v}")

    return fp_words, fp_by_type


In [69]:
fp_words, fp_by_type = analyze_false_positives(all_preds, all_labels, val_sentences)



🔴 Top False Positive Tokens:
für                       6
im                        2
Liga                      2
und                       2
die                       2
BA                        2
.                         2
and                       2
Mastersstand              1
Kulturzentrum             1
Schlachthof               1
an                        1
der                       1
Fernsehen                 1
Finanzberatung            1
Victor                    1
Happy                     1
Dinos                     1
Ponyclub                  1
Lanterne                  1
Bürger                    1
Fustanella                1
Kohlenstoffdioxid         1
OT                        1
Forum                     1
Ahltener                  1
Wald                      1
Literatur                 1
um                        1
acht                      1

📊 FP by Entity Type:
ORG: 111
LOC: 31
PER: 28


In [79]:
import torch
state = torch.load("submission/weights/head.pt", map_location="cpu")
print([k for k in state.keys() if "crf" in k.lower()])


['crf.start_transitions', 'crf.end_transitions', 'crf.transitions']


In [87]:
import zipfile

ZIP_PATH = "submission.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

print("ZIP FILE COUNT:", len(names))
print("\n--- ZIP CONTENTS ---")
for n in names:
    print(n)

required = {
    "model.py",
    "weights/head.pt",
    "weights/encoder/config.json",
    "weights/encoder/model.safetensors",
    "weights/encoder/tokenizer.json",
}

print("\n--- REQUIRED CHECK ---")
missing = [r for r in required if r not in names]
if missing:
    print("❌ Missing required files:")
    for m in missing:
        print("  ", m)
else:
    print("✅ All required files are present.")


ZIP FILE COUNT: 8

--- ZIP CONTENTS ---
model.py
weights/head.pt
weights/encoder/model.safetensors
weights/encoder/tokenizer_config.json
weights/encoder/special_tokens_map.json
weights/encoder/config.json
weights/encoder/tokenizer.json
weights/encoder/vocab.txt

--- REQUIRED CHECK ---
✅ All required files are present.


In [88]:
import os, shutil, zipfile
import numpy as np
import importlib.util
import sys

ZIP_PATH = "submission.zip"
TMP_DIR = "_tmp_check_submission"

# clean + unzip
if os.path.exists(TMP_DIR):
    shutil.rmtree(TMP_DIR)
os.makedirs(TMP_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(TMP_DIR)

# dynamic import model.py from the extracted folder
model_path = os.path.join(TMP_DIR, "model.py")
spec = importlib.util.spec_from_file_location("submitted_model", model_path)
submitted = importlib.util.module_from_spec(spec)
sys.modules["submitted_model"] = submitted
spec.loader.exec_module(submitted)

M = submitted.Model()
print("✅ Model loaded from zip.")


✅ Model loaded from zip.


In [89]:
allowed = {"O","B-PER","I-PER","B-ORG","I-ORG","B-LOC","I-LOC"}

tests = [
    np.array(["Muck", "wird", "Schatzmeister", "."], dtype=object),
    np.array(["1951","bis","1953","wurde","der","nördliche","Teil","gebaut","."], dtype=object),
    np.array([], dtype=object),
    np.array(["A"]*200, dtype=object),  # stress: long sequence
]

for i, x in enumerate(tests, 1):
    y = M.predict(x)
    ok_len = (len(y) == len(x))
    ok_labels = all(lbl in allowed for lbl in y.tolist())
    print(f"Test {i}: n_in={len(x)}  n_out={len(y)}  len_ok={ok_len}  labels_ok={ok_labels}")

    if not ok_len:
        print("❌ Length mismatch example:", x[:10], y[:10])
    if not ok_labels:
        bad = [lbl for lbl in y.tolist() if lbl not in allowed][:10]
        print("❌ Bad labels example:", bad)


Test 1: n_in=4  n_out=4  len_ok=True  labels_ok=True
Test 2: n_in=9  n_out=9  len_ok=True  labels_ok=True
Test 3: n_in=0  n_out=0  len_ok=True  labels_ok=True
Test 4: n_in=200  n_out=200  len_ok=True  labels_ok=True


In [ ]:
import torch

state = torch.load("submission/weights/head.pt", map_location="cpu")

# transitions CRF
t = state["crf.transitions"]
print("transitions mean/std:", t.mean().item(), t.std().item())

#classifier weights
w = state["classifier.weight"]
print("classifier mean/std:", w.mean().item(), w.std().item())


transitions mean/std: -0.2865890562534332 0.5216172933578491
classifier mean/std: -0.0007343007600866258 0.04830586537718773


In [ ]:
import torch

a = torch.load("best_head_weighted.pt", map_location="cpu")         
b = torch.load("submission/weights/head.pt", map_location="cpu")     

diff = (a["classifier.weight"] - b["classifier.weight"]).abs().mean().item()
diff2 = (a["crf.transitions"] - b["crf.transitions"]).abs().mean().item()

print("mean abs diff classifier.weight:", diff)
print("mean abs diff crf.transitions:", diff2)


mean abs diff classifier.weight: 0.0
mean abs diff crf.transitions: 0.0
